# InstructBLIP (Flan-T5-XL) Image Phrase Extractor — Google Colab (Google Drive)

**Model:** `Salesforce/instructblip-flan-t5-xl` — publicly available, no HuggingFace login required.

**Images are read directly from Google Drive.** No ZIP upload needed.

### Before running:

1. Go to [drive.google.com](https://drive.google.com) and upload your image folder into **My Drive**
2. Set `DRIVE_IMAGE_FOLDER` in Cell 1 to the exact folder name you uploaded
3. **Allow popups** for `colab.research.google.com` in your browser — the Drive mount opens a Google sign-in popup

**If Cell 1 fails with a credential error:**
- Allow popups in your browser settings and re-run Cell 1
- Or try: `Runtime → Restart session` then re-run

**Runtime recommendation:** `Runtime → Change runtime type → T4 GPU` (free tier) — model downloads ~4 GB on first run.

In [ ]:
# ── Cell 1: Mount Google Drive and set image folder ───────────────────────────
from google.colab import auth, drive
import os, zipfile

# Explicitly authenticate with your Google account first (opens a popup)
auth.authenticate_user()

# Then mount Drive
drive.mount('/content/drive', force_remount=True)

# ↓ Change this to the name of the folder you uploaded to My Drive
DRIVE_IMAGE_FOLDER = 'vqaimages'

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_IMAGE_FOLDER}'

assert os.path.isdir(DRIVE_DIR), (
    f'Folder not found: {DRIVE_DIR}\n'
    f'Make sure "{DRIVE_IMAGE_FOLDER}" exists inside My Drive on Google Drive.'
)

# If the folder contains ZIP files, extract them locally
zip_files = [f for f in os.listdir(DRIVE_DIR) if f.lower().endswith('.zip')]
if zip_files:
    EXTRACT_DIR = f'/content/{DRIVE_IMAGE_FOLDER}_extracted'
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    for zf in zip_files:
        print(f'Extracting {zf} ...')
        with zipfile.ZipFile(os.path.join(DRIVE_DIR, zf), 'r') as z:
            z.extractall(EXTRACT_DIR)
    IMAGE_DIR = EXTRACT_DIR
    print(f'Extracted to {IMAGE_DIR}')
else:
    IMAGE_DIR = DRIVE_DIR
    print(f'Reading images directly from {IMAGE_DIR}')

total = sum(1 for _, _, fs in os.walk(IMAGE_DIR) for f in fs)
print(f'Found {total} file(s) in total')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers accelerate Pillow opencv-python-headless numpy torch torchvision

In [ ]:
# ── Cell 3: Load InstructBLIP (Flan-T5-XL) model ─────────────────────────────
# Downloads ~4 GB on first run — cached for the rest of the session.
import torch
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration

MODEL_ID = 'Salesforce/instructblip-flan-t5-xl'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE    = torch.float16 if DEVICE == 'cuda' else torch.float32

print(f'Loading {MODEL_ID} on {DEVICE.upper()} ...')
processor = InstructBlipProcessor.from_pretrained(MODEL_ID)
model     = InstructBlipForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE,
).eval()
print(f'Model ready  |  device: {DEVICE.upper()}  |  dtype: {DTYPE}')

In [ ]:
# ── Cell 4: Configuration ─────────────────────────────────────────────────────
MAX_IMAGE_PX     = 512
MAX_NEW_TOKENS   = 40
FRAMES_PER_VIDEO = 4

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.wmv', '.m4v'}

PROMPT = (
    'Describe this image in one short phrase focusing on '
    'whether fire is present, its nature (e.g. wildfire, campfire, industrial flame, explosion), '
    'and the surrounding environment. '
    'Reply with the phrase only.'
)

In [ ]:
# ── Cell 5: Helper functions ──────────────────────────────────────────────────
import time
from pathlib import Path

import cv2
import numpy as np
from PIL import Image


def resize_image(img, max_px=MAX_IMAGE_PX):
    w, h = img.size
    if max(w, h) <= max_px:
        return img
    scale = max_px / max(w, h)
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)


def get_phrase(img):
    img    = resize_image(img.convert('RGB'))
    inputs = processor(images=img, text=PROMPT, return_tensors='pt').to(DEVICE)
    if DTYPE == torch.float16:
        inputs = {k: v.half() if v.dtype == torch.float32 else v for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )
    return processor.decode(output_ids[0], skip_special_tokens=True).strip()


def extract_frames(video_path, n=FRAMES_PER_VIDEO):
    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return []
    frames = []
    for idx in np.linspace(0, total - 1, n, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames


print('Helper functions ready.')

In [ ]:
# ── Cell 6: Process all images / videos ───────────────────────────────────────
root     = Path(IMAGE_DIR)
out_path = Path('/content/drive/MyDrive/phrases_instructblip.txt')  # saved to Drive

all_files = sorted(
    p for p in root.rglob('*')
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS | VIDEO_EXTS
)

print(f'Found {len(all_files)} file(s)\n' + '─' * 60)

with open(out_path, 'w') as out:
    for idx, path in enumerate(all_files, 1):
        rel = path.relative_to(root)
        ext = path.suffix.lower()
        print(f'[{idx}/{len(all_files)}] {rel}')

        try:
            t0 = time.time()

            if ext in IMAGE_EXTS:
                phrase = get_phrase(Image.open(path))
                print(f'  → {phrase!r}  ({time.time()-t0:.1f}s)')
                out.write(f'{rel} | {phrase}\n')

            else:
                frames = extract_frames(path)
                for i, frame in enumerate(frames):
                    phrase = get_phrase(frame)
                    print(f'  frame {i+1}/{len(frames)} → {phrase!r}  ({time.time()-t0:.1f}s)')
                    out.write(f'{rel} [frame {i+1}] | {phrase}\n')
                    t0 = time.time()

        except Exception as e:
            print(f'  ERROR: {e}')
            out.write(f'{rel} | ERROR: {e}\n')

        out.flush()

print(f'\nDone. phrases_instructblip.txt saved to My Drive')